[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/session2_logistic_map_bridge.ipynb)


# The Logistic Map — What Kollman & Page's Corpus Leaves Out

Kollman & Page (2006) give us seven concepts for social complexity: adaptation, difference,
externalities, path dependence, geography, networks, emergence. All seven assume something in
common -- **many agents, interacting**.

This notebook is about a different, older lineage of complexity science that needs none of that:
a single number, one simple equation, no agents, no interaction, no randomness at all --

$$x_{t+1} = r \cdot x_t (1 - x_t)$$

the **logistic map**. And yet it produces genuine unpredictability. Understanding *why* fills a
real gap between Weaver/Simon and K&P, and gives you a second, completely different mechanism for
complexity to compare against everything else in this course.

**The throughline for this notebook, in your own course's vocabulary:**
- Weaver sorted problems by *variable count*. The logistic map has exactly **one** variable and
  one parameter -- by cardinality alone it should be the simplest possible case. It isn't.
- What actually matters is **rules** (linear vs. nonlinear) and **feedback type** (homeostatic
  vs. generative) -- not how many variables are in the system.
- Simon's bounded rationality doesn't even apply here -- there's no agent to be bounded. This is
  the cleanest possible demonstration that unpredictability doesn't require agents at all.
- K&P's "emergence" and this notebook's "emergence" will turn out to be **the same word for two
  different mechanisms** -- worth being precise about, not blurring together.

**How this notebook is presented:** same house style as everything else in the course --
pseudocode cell, then a matching numbered code cell, right below it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Ready.")


---
## 1. Nonlinearity — the one variable that isn't simple

By Weaver's cardinality-based sorting, one variable and one parameter should mean "simplicity,"
his easiest category. But the logistic map's rule is **nonlinear**: output isn't proportional to
input, and there's no fixed, constant relationship you can write as `y = a*x + b`.


### Pseudocode — `logistic_step`

1. Take the current value `x` and the parameter `r`
2. Return `r * x * (1 - x)` -- the next value


In [ ]:
def logistic_step(x, r):
    """One step of the logistic map: how much x changes depends on x itself,
    not just on a fixed rate -- that's what makes this nonlinear."""
    return r * x * (1 - x)   # (1), (2)


### Does a straight line describe this rule? Let's check with the simplest possible fit: OLS.

This matters beyond the logistic map itself -- it's a small, concrete version of a bigger warning
this course keeps returning to: **a model can fit well and still have recovered nothing about the
underlying mechanism.**


In [ ]:
x_grid = np.linspace(0, 1, 400)
r_value = 0.8
y_true = logistic_step(x_grid, r_value)

# simplest possible linear fit: y = slope * x + intercept
slope, intercept = np.polyfit(x_grid, y_true, deg=1)
y_linear_fit = slope * x_grid + intercept

plt.figure()
plt.plot(x_grid, y_true, linewidth=3, label="True rule: logistic step")
plt.plot(x_grid, y_linear_fit, linewidth=2, linestyle="--", label="Best straight-line fit")
plt.title("A straight line cannot describe a nonlinear rule")
plt.xlabel("$x_t$"); plt.ylabel("$x_{t+1}$")
plt.legend()
plt.show()


**Takeaway:** the straight line misses the shape entirely -- it's not a subtle miss, it's the
wrong family of function altogether. A flexible enough curve-fitter (a small neural network, a
high-degree polynomial) *can* trace this curve closely. But even a perfect trace only recovers the
**shape** of the rule at `r=0.8` -- it never recovers `r` itself, the actual structural parameter
generating the curve. Fit it on data from one `r` and hand it data from a different `r`, and it
has no idea anything changed. That's the difference between fitting a shadow and recovering a
mechanism -- keep this question in mind for every model you build for the rest of this course:
**did I recover the mechanism, or just its shadow?**


---
## 2. Attractors — where the system settles, and what "settling" implies about memory

Iterate the map many times from a starting point, and (at low `r`) the sequence settles onto one
fixed value -- an **attractor**. In your framework's terms: once a trajectory settles onto a
single attractor regardless of exactly where it started, the system has effectively **forgotten**
its initial condition -- long-run behavior becomes **path-independent (Markov)**, not path-
dependent.


### Pseudocode — `logistic_series`

1. Start a list with the initial value `x0`
2. Repeat `steps - 1` times: apply `logistic_step` to the last value, and add the result to the
   list
3. Return the whole sequence as an array


In [ ]:
def logistic_series(r, x0, n_steps):
    """Repeatedly apply the logistic map, starting from x0, and return
    every value visited along the way."""
    xs = [x0]                                    # (1)
    for _ in range(n_steps - 1):                  # (2)
        xs.append(logistic_step(xs[-1], r))
    return np.array(xs)                           # (3)


In [ ]:
r_value = 2.5
series = logistic_series(r=r_value, x0=0.1, n_steps=40)
x_star = 1 - 1/r_value   # the fixed point, solved analytically: x* = r*x*(1-x*)

plt.figure()
plt.plot(series, marker="o", markersize=3)
plt.axhline(x_star, color="red", linestyle="--", label=f"attractor, $x^*={x_star:.3f}$")
plt.title(f"Settling onto a fixed point (r={r_value})")
plt.xlabel("step"); plt.ylabel("$x_t$")
plt.legend()
plt.show()


### A quick, static cobweb diagram -- the geometric picture of "settling"

Each step is: go up/down to the curve (apply the rule), then across to the diagonal (feed the
result back in as the next `x`). Spiraling in toward one point *is* what "attractor" looks like
geometrically.


In [ ]:
def cobweb_lines(r, x0, n_steps):
    """Build the staircase of (x, y) points that trace a cobweb diagram --
    alternating between the curve and the diagonal."""
    x = x0
    points_x, points_y = [x], [0]
    for _ in range(n_steps):
        y = logistic_step(x, r)
        points_x += [x, y]     # up/down to the curve
        points_y += [y, y]
        x = y
        points_x += [x]        # across to the diagonal
        points_y += [x]
    return points_x, points_y

cx, cy = cobweb_lines(r=2.5, x0=0.1, n_steps=25)

plt.figure(figsize=(5.5, 5.5))
xs = np.linspace(0, 1, 300)
plt.plot(xs, logistic_step(xs, 2.5), "k", label="rule")
plt.plot(xs, xs, "k--", alpha=0.5, label="$x_{t+1}=x_t$")
plt.plot(cx, cy, color="tab:red", linewidth=1, label="trajectory (cobweb)")
plt.title("Cobweb diagram: spiraling into the attractor")
plt.legend()
plt.show()


---
## 3. More than one attractor -- and a word of caution about "emergence"

Raise `r` far enough, and the system stops settling on *one* value -- it starts alternating
between 2, then 4, then 8 values, then eventually never repeats at all (chaos). This qualitative
jump, from one attractor to many, as `r` changes smoothly, is sometimes also called **emergence**
in the complexity-science literature.

**Be precise here, because this is not K&P's emergence.** K&P's emergence (Epstein's rebellion
waves) is a surprising *aggregate pattern from many interacting agents*. This is a *qualitative
change in a single deterministic equation's long-run behavior* as one parameter crosses a
threshold -- no agents, no population, no interaction at all. **Same word, two different
mechanisms.** In your framework's terms: this is what **generative feedback** produces -- feedback
that builds new structure, rather than **homeostatic feedback**, which just restores a target
state.


### Pseudocode — `sweep_bifurcation` (vectorized -- every `r` at once, no per-`r` Python loop)

1. Build an array of many `r` values to test at once
2. Start every `r`'s trajectory from the same `x0`, all as one array
3. Run the map forward `n_transient` times (as one array operation each step) and throw these away
   -- this lets each `r`'s trajectory settle onto its actual long-run attractor first
4. Run forward `n_keep` more times, and this time record every value -- these recorded values are
   the attractor(s) for that `r`
5. Return the `r` values and their corresponding long-run `x` values, ready to scatter-plot


In [ ]:
def sweep_bifurcation(r_min=2.5, r_max=4.0, n_r=2000, n_transient=300, n_keep=150, x0=0.2):
    """For many r values at once, discard the early transient, then record the
    long-run attractor values. Fully vectorized -- one array per step, not one
    Python loop per r."""
    r_values = np.linspace(r_min, r_max, n_r)                     # (1)
    x = np.full(n_r, x0)                                          # (2)

    for _ in range(n_transient):                                  # (3)
        x = logistic_step(x, r_values)

    recorded_r, recorded_x = [], []
    for _ in range(n_keep):                                       # (4)
        x = logistic_step(x, r_values)
        recorded_r.append(r_values)
        recorded_x.append(x)

    return np.concatenate(recorded_r), np.concatenate(recorded_x)  # (5)


In [ ]:
bif_r, bif_x = sweep_bifurcation()

plt.figure()
plt.scatter(bif_r, bif_x, s=0.15, alpha=0.4, color="black")
plt.title("Bifurcation diagram: how many attractors, at each r")
plt.xlabel("r"); plt.ylabel("long-run $x_t$ values")
plt.show()


**What to read off this plot:** one line at low `r` (one attractor) -- then it splits into two,
then four, then eight, faster and faster, then dissolves into a dense scatter (chaos). No
randomness was used anywhere in `sweep_bifurcation` -- every bit of this structure is a direct,
deterministic consequence of one equation and one changing parameter.


---
## 4. Sensitivity to initial conditions -- the mechanism, contrasted with Session 1's Case 3

In the chaotic region, two starting points that differ by almost nothing end up on completely
different trajectories. This is a **second, distinct mechanism for unpredictability** --
distinct from what made Session 1's Case 3 (Model B) unpredictable:

| | Case 3 (Session 1) | This notebook |
|---|---|---|
| Actors | many walkers | none -- one number |
| Randomness | yes (random tie-breaks, random seeds) | none -- fully deterministic |
| Source of unpredictability | history + social feedback | exponential amplification of tiny starting differences |

Both produce "can't predict the outcome from the parameters alone" -- for completely different
reasons. Don't let the shared symptom collapse into a shared cause.


### Pseudocode — `two_trajectories`

1. Start two runs of the map at nearly the same `x0`, differing only by a tiny amount `epsilon`
2. Advance both, step by step, using the identical rule and the identical `r`
3. Return both full trajectories


In [ ]:
def two_trajectories(r, x0=0.2, epsilon=1e-8, n_steps=60):
    """Run two nearly-identical starting points through the same rule,
    and return both trajectories so we can compare them."""
    x1, x2 = [x0], [x0 + epsilon]                 # (1)
    for _ in range(n_steps - 1):                   # (2)
        x1.append(logistic_step(x1[-1], r))
        x2.append(logistic_step(x2[-1], r))
    return np.array(x1), np.array(x2)              # (3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for r_val, ax in zip([2.9, 3.9], axes):
    x1, x2 = two_trajectories(r=r_val)
    ax.plot(x1, label="$x_0$")
    ax.plot(x2, linestyle="--", label="$x_0+\\epsilon$")
    ax.set_title(f"r = {r_val}")
    ax.legend()

plt.suptitle("Two nearly-identical starting points")
plt.tight_layout()
plt.show()


**What to look for:** at `r=2.9`, the two lines merge -- the system forgets the tiny initial
difference. At `r=3.9`, they track together briefly, then diverge completely -- the system
amplifies and "remembers" that tiny difference, blowing it up to macroscopic size.


### Pseudocode — `lyapunov_exponent` (analytical, exact -- not estimated from data)

Because we know the logistic map's exact equation, we don't need to estimate this from noisy
trajectory data -- we can compute it directly from the map's own derivative.

1. Start at `x0`, and run the map forward `burn_in` times without recording anything (let
   transients settle out)
2. From there, repeat `n_steps` more times:
   1. Compute the derivative of the map at the current `x`: `|r * (1 - 2x)|`
   2. Take its logarithm and add it to a running total
   3. Advance `x` to the next step
3. Return the average of all those logged derivatives -- this average *is* the Lyapunov exponent
   `λ`


In [ ]:
def lyapunov_exponent(r, x0=0.4, burn_in=200, n_steps=2000):
    """Compute the Lyapunov exponent directly from the map's own derivative --
    exact, because we know the equation, not an approximation from data."""
    x = x0
    for _ in range(burn_in):                        # (1)
        x = logistic_step(x, r)

    log_derivative_sum = 0.0
    for _ in range(n_steps):                         # (2)
        derivative = abs(r * (1 - 2*x))               # (2.1)
        log_derivative_sum += np.log(derivative + 1e-16)  # (2.2)
        x = logistic_step(x, r)                        # (2.3)

    return log_derivative_sum / n_steps               # (3)


### Sweep `λ(r)` across the same range as the bifurcation diagram, and line them up


In [ ]:
r_sweep = np.linspace(2.5, 4.0, 400)
lyapunov_values = np.array([lyapunov_exponent(r) for r in r_sweep])

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].scatter(bif_r, bif_x, s=0.1, alpha=0.3, color="black")
axes[0].set_ylabel("long-run $x_t$")
axes[0].set_title("Bifurcation diagram (top) vs. Lyapunov exponent (bottom)")

axes[1].plot(r_sweep, lyapunov_values, color="tab:red")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("$\\lambda(r)$")
axes[1].set_xlabel("r")
plt.tight_layout()
plt.show()


**Reading `λ(r)`:**

| Value | Meaning |
|---|---|
| `λ(r) < 0` | stable fixed point or cycle -- errors shrink, prediction is reliable |
| `λ(r) = 0` | critical boundary -- right at a bifurcation |
| `λ(r) > 0` | chaos -- errors grow exponentially, long-run prediction breaks down |

**Two concrete cases, worth sitting with:**
- **`r = 2.9`** (`λ < 0`): each step *shrinks* a small error by about 10%. A deviation is cut in
  half after roughly 6 steps, and down to one-tenth its original size after about 22 steps.
  Long-run prediction is genuinely reliable here.
- **`r = 3.7`** (`λ > 0`): each step *amplifies* a small error by about 43%. Any initial
  uncertainty has already doubled in under 2 steps -- meaning any real intervention has to happen
  almost immediately -- and by 6 steps, errors are an order of magnitude larger than they started.
  Past that point, precise long-term forecasting is not a data problem or a modeling-skill
  problem. It's mathematically impossible, for this system, no matter how good your model or your
  data are.

That last point is worth sitting with as a general lesson, independent of the logistic map
itself: sometimes the honest answer to "can we predict this more precisely" is *no, not with any
amount of better data or better modeling* -- past a chaotic threshold, the right move shifts from
prediction to adaptive management, responding to what actually happens rather than betting on a
forecast.


---
## Closing: what this hands off to K&P

**What this notebook covered that K&P's seven concepts never touch:**
- Nonlinearity as the real driver of unpredictability -- not variable count (corrects an implicit
  gap in Weaver's own cardinality-based sorting)
- Attractors, and what "settling" implies about memory (path-independent / Markov, in your
  framework's terms)
- A second sense of "emergence" (bifurcation) -- explicitly distinguished from K&P's own,
  agent-based sense of the word
- A second, distinct mechanism for unpredictability (deterministic chaos / sensitivity to initial
  conditions) -- contrasted directly against Session 1 Case 3's mechanism (stochastic history +
  social feedback)

**Where this connects to what's coming:**
- K&P's whole seven-concept corpus assumes *many interacting agents* -- everything in this
  notebook assumed the opposite: one variable, zero agents, zero interaction. Both traditions
  independently arrived at "simple rules can produce genuine unpredictability" -- by completely
  different routes, decades apart (this lineage: Poincaré's analytical glimpse in the 1890s, made
  computationally visible by Lorenz in 1963; K&P's lineage: agent-based social science, maturing
  through the Santa Fe Institute from the 1980s onward).
- The standing question from Section 1 -- **did I recover the mechanism, or just its shadow?** --
  carries forward into every model this course builds from here, agent-based or not.
